# 1. Upload Image Dataset and Prepare Training Data

#### 1.1 Copy dataset zip from google drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

!cp /content/gdrive/MyDrive/label_studio_dataset.zip /content

#### 1.2 Split images into train and validation folders



In [ ]:
!unzip -q /content/label_studio_dataset.zip -d /content/label_studio_dataset

In [ ]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

!python train_val_split.py --datapath="/content/label_studio_dataset/label_studio_dataset" --train_pct=0.9

# 2. Install Requirements


In [ ]:
!pip install ultralytics

# 3. Configure Training

In [ ]:
import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/label_studio_dataset/label_studio_dataset/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

# 4. Train and Export Model

#### 4.1 Training model

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo26n.pt epochs=100 imgsz=320

#### 4.2 Export to onnx

In [ ]:
!yolo export model=/content/runs/detect/train/weights/best.pt format=onnx

# 5. Post Training Quantization to INT8

#### 5.1 Optimizing onnx model


In [ ]:
from onnxruntime.quantization import shape_inference

shape_inference.quant_pre_process(
    input_model_path="models/best.onnx",
    output_model_path="models/best-preprocessed.onnx",
)

print("✅ Saved: models/best-preprocessed.onnx")

#### 5.2 Quantizing the model

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
import onnx
from onnxruntime.quantization import (
    quantize_static,
    QuantType,
    QuantFormat,
    CalibrationDataReader,
    CalibrationMethod,
)
import onnxruntime as ort


# ============================================================
# CONFIGURATION
# ============================================================
PREPROCESSED_MODEL = "models/best-preprocessed.onnx"
OUTPUT_MODEL = "models/yolo26n_fine_tuned_int8.onnx"
CALIBRATION_DIR = "data/validation/images"
IMGSZ = 320
MAX_CALIBRATION_IMAGES = 200
# ============================================================


class YOLOCalibrationReader(CalibrationDataReader):
    def __init__(self, calibration_dir, input_name, imgsz, max_images):
        self.imgsz = imgsz
        self.input_name = input_name
        self.enum_data = None

        valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
        self.image_paths = sorted([
            p for p in Path(calibration_dir).iterdir()
            if p.suffix.lower() in valid_extensions
        ])
        self.image_paths = self.image_paths[:max_images]
        print(f"[Calibration] Found {len(self.image_paths)} images in {calibration_dir}")

        if len(self.image_paths) == 0:
            raise FileNotFoundError(f"No images found in {calibration_dir}.")

        self.enum_data = iter(self._load_all())

    def _load_all(self):
        data_list = []
        for i, img_path in enumerate(self.image_paths):
            img = Image.open(img_path).convert("RGB")
            img = img.resize((self.imgsz, self.imgsz), Image.BILINEAR)
            img_np = np.array(img, dtype=np.float32) / 255.0
            img_np = np.transpose(img_np, (2, 0, 1))
            img_np = np.expand_dims(img_np, axis=0)
            data_list.append({self.input_name: img_np})
            if (i + 1) % 50 == 0:
                print(f"  Loaded {i + 1}/{len(self.image_paths)} calibration images...")
        return data_list

    def get_next(self):
        return next(self.enum_data, None)

    def rewind(self):
        self.enum_data = iter(self._load_all())


# ============================================================
# 1. FIND NODES TO EXCLUDE
# ============================================================
print("Scanning model for sensitive nodes...\n")
model = onnx.load(PREPROCESSED_MODEL)

# These blocks contain ops that break under INT8:
#   /model.9/  — SPPF pooling block
#   /model.10/ — A2C2f attention block in neck
#   /model.22/ — A2C2f attention block (MatMul, Softmax)
#   /model.23/ — Detection head + post-processing (TopK, Mod, GatherElements, Cast)
EXCLUDE_PREFIXES = ["/model.9/", "/model.10/", "/model.22/", "/model.23/"]

nodes_to_exclude = []
for node in model.graph.node:
    if any(prefix in node.name for prefix in EXCLUDE_PREFIXES):
        nodes_to_exclude.append(node.name)

total_nodes = len(model.graph.node)
quantized_nodes = total_nodes - len(nodes_to_exclude)

print(f"Total nodes in model    : {total_nodes}")
print(f"Nodes excluded (FP32)   : {len(nodes_to_exclude)}")
print(f"Nodes quantized (INT8)  : {quantized_nodes}")
print(f"Quantization coverage   : {quantized_nodes/total_nodes*100:.1f}%\n")

del model

# ============================================================
# 2. GET MODEL INPUT NAME
# ============================================================
session = ort.InferenceSession(PREPROCESSED_MODEL)
input_name = session.get_inputs()[0].name
print(f"Model input name: {input_name}")
del session

# ============================================================
# 3. CREATE CALIBRATION READER
# ============================================================
calibration_reader = YOLOCalibrationReader(
    calibration_dir=CALIBRATION_DIR,
    input_name=input_name,
    imgsz=IMGSZ,
    max_images=MAX_CALIBRATION_IMAGES,
)

# ============================================================
# 4. QUANTIZE
# ============================================================
print("\nStarting INT8 quantization...\n")

quantize_static(
    model_input=PREPROCESSED_MODEL,
    model_output=OUTPUT_MODEL,
    calibration_data_reader=calibration_reader,
    quant_format=QuantFormat.QDQ,
    weight_type=QuantType.QInt8,
    activation_type=QuantType.QUInt8,
    per_channel=True,
    calibrate_method=CalibrationMethod.Entropy,
    nodes_to_exclude=nodes_to_exclude,
)

# ============================================================
# 5. RESULTS
# ============================================================
original_size = Path(PREPROCESSED_MODEL).stat().st_size / (1024 * 1024)
quantized_size = Path(OUTPUT_MODEL).stat().st_size / (1024 * 1024)

print(f"\n✅ Quantization complete!")
print(f"   Output: {OUTPUT_MODEL}")
print(f"   FP32 model : {original_size:.2f} MB")
print(f"   INT8 model : {quantized_size:.2f} MB")
print(f"   Compression: {original_size / quantized_size:.1f}x smaller")

#### 5.3 Validating the INT8 quantized onnx model

In [ ]:
import onnxruntime as ort
import numpy as np
from pathlib import Path
from PIL import Image
import time

IMGSZ = 320
FP32_MODEL = "models/best-preprocessed.onnx"
INT8_MODEL = "models/yolo26n_fine_tuned_int8.onnx"
VALIDATION_DIR = "data/validation/images"


def preprocess(image_path):
    img = Image.open(image_path).convert("RGB").resize((IMGSZ, IMGSZ))
    img_np = np.array(img, dtype=np.float32) / 255.0
    img_np = np.transpose(img_np, (2, 0, 1))
    return np.expand_dims(img_np, axis=0)


# Load both models
fp32_session = ort.InferenceSession(FP32_MODEL)
int8_session = ort.InferenceSession(INT8_MODEL)
input_name = fp32_session.get_inputs()[0].name

# Grab validation images
test_images = sorted([
    p for p in Path(VALIDATION_DIR).iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
])

# ── Accuracy Check ──
print("--- Accuracy Check (FP32 vs INT8) ---\n")
diffs = []
for img_path in test_images[:20]:
    test_img = preprocess(img_path)
    fp32_out = fp32_session.run(None, {input_name: test_img})
    int8_out = int8_session.run(None, {input_name: test_img})

    mean_diff = np.mean(np.abs(fp32_out[0] - int8_out[0]))
    max_diff = np.max(np.abs(fp32_out[0] - int8_out[0]))
    diffs.append(mean_diff)
    print(f"  {img_path.name[:45]:45s} mean={mean_diff:.6f}  max={max_diff:.6f}")

avg = np.mean(diffs)
print(f"\n  Average mean diff across {len(diffs)} images: {avg:.6f}")

if avg < 1.0:
    print("  ✅ GREAT — quantization is healthy!")
elif avg < 5.0:
    print("  ⚠️  OKAY — minor accuracy drop, should work fine")
else:
    print("  ❌ Too high — need to exclude more nodes in quantize_int8.py")

# ── Speed Benchmark ──
print("\n--- Speed Benchmark (laptop) ---\n")
test_img = preprocess(test_images[0])

for name, sess in [("FP32", fp32_session), ("INT8", int8_session)]:
    for _ in range(10):
        sess.run(None, {input_name: test_img})
    runs = 100
    start = time.time()
    for _ in range(runs):
        sess.run(None, {input_name: test_img})
    elapsed = time.time() - start
    print(f"  {name}: {(elapsed / runs) * 1000:.1f} ms/frame  ({runs / elapsed:.1f} FPS)")

# ── File Sizes ──
fp32_size = Path(FP32_MODEL).stat().st_size / (1024 * 1024)
int8_size = Path(INT8_MODEL).stat().st_size / (1024 * 1024)
print(f"\n  FP32: {fp32_size:.2f} MB")
print(f"  INT8: {int8_size:.2f} MB")
print(f"  Compression: {fp32_size / int8_size:.1f}x smaller")